Rita Fadhel, ECE284 Final Project, CNN + LSTM hybrid on the HAM10000 dataset


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder


import torchvision.transforms as transforms
import torch.nn.functional as F

from PIL import Image
import os
import numpy as np
import matplotlib.pyplot as plt


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
# Image preprocessing, augmentation

# Image preprocessing (matches paper)
transform = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor()
])

dataset = ImageFolder("HAM10000_dataset_path", transform=transform)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True)


In [ ]:
# patch function

def image_to_patches(images, patch_size=10):
    B, C, H, W = images.shape #batch size, num of channels (3 RGB), height&width

    #extract patches along height and width 
    patches = images.unfold(2, patch_size, patch_size).unfold(3, patch_size, patch_size)
    patches = patches.contiguous().view(B, C, -1, patch_size, patch_size) 
    patches = patches.permute(0, 2, 1, 3, 4)  
    return patches

In [ ]:
#Hybrid LSTM-CNN model
class LSTM_CNN_Model(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        self.patch_size = 10
        self.input_dim = 3 * self.patch_size * self.patch_size #patches flattened before LSTM
        # 3channels * 10 * 10 = 300 features per patch

        # LSTM 
        self.lstm = nn.LSTM(
            input_size=self.input_dim, #size of each patch
            hidden_size=128, #numb of featured learned per timestep
            batch_first=True
        )

        # CNN applied per timestep (Time-Distributed)
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), #spatial size
            nn.ReLU(),                                  #non-linear
            nn.MaxPool2d(2),                            #downsample
            nn.Conv2d(16, 32, kernel_size=3, padding=1), #deep feature extraction
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.dropout = nn.Dropout(0.4) #paper uses 0.3-0.5
        self.fc = nn.Linear(32 * 4 * 4, num_classes) # 32*4*4 = 512 features

    def forward(self, x):
        # x: (B, C, H, W)
        patches = image_to_patches(x)  #output: (B, N, C, p, p)
        B, N, C, p, _ = patches.shape

        # Flattening patches for LSTM
        patches = patches.view(B, N, -1)

        # LSTM
        lstm_out, _ = self.lstm(patches)  # (B, N, hidden)

        # Applying CNN per timestep
        outputs = []
        for t in range(N):
            step = lstm_out[:, t, :]  # (B, hidden)

            # reshaping to pseudo-image
            step = step.unsqueeze(1).unsqueeze(-1)
            step = step.repeat(1, 1, 16, 16)  # reshape to 2D

            out = self.conv(step)
            out = out.view(out.size(0), -1)
            outputs.append(out)

        outputs = torch.stack(outputs, dim=1).mean(dim=1)

        outputs = self.dropout(outputs)
        outputs = self.fc(outputs)

        return outputs


In [ ]:
#Training loop

model = LSTM_CNN_Model().cuda()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-3)

for epoch in range(30):
    model.train()
    for images, labels in train_loader:
        images, labels = images.cuda(), labels.cuda()

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch} Loss: {loss.item()}")